# Segundo revisor por embeddings — detector de discrepancias

**Proyecto BME513 · verificación de utilidad**

Este notebook implementa y **mide** un *segundo revisor* independiente para la
clasificación de recomendaciones. No reemplaza a las reglas ni decide coherencia: corre
en paralelo, y **cuando discrepa de las reglas, marca el caso para revisión humana**.

Principios de diseño (acordados):
- **Ciego al BI-RADS:** clasifica solo desde el texto de la recomendación (evita
  circularidad: el BI-RADS solo se usa para *evaluar*, nunca para clasificar).
- **Independiente:** usa embeddings semánticos (método distinto al de las reglas), para
  que su acuerdo/desacuerdo aporte información real.
- **Sin ruido conocido:** no dispara discrepancia cuando el desacuerdo es solo entre las
  dos categorías de imagen que se solapan (correlación ↔ estudio complementario), donde
  ya sabemos que el embedding no es fiable.

Qué mide este notebook:
1. **Tasa de discrepancia útil** sobre el corpus (¿cuántas alarmas генera, y son ruido?).
2. **Cobertura en casos atípicos** (¿detecta recomendaciones con sinónimos que las reglas
   fallarían?).
3. **Veredicto:** ¿aporta al proyecto o genera demasiado ruido?


## 1. Instalación y modelo

In [1]:
!pip install -q -U sentence-transformers
import numpy as np, pandas as pd, io
from sentence_transformers import SentenceTransformer, models

# Modelo clínico en español (el mejor en las pruebas previas: 7/8 en atípicas)
def cargar_clinico(nombre="PlanTL-GOB-ES/bsc-bio-ehr-es"):
    w = models.Transformer(nombre, max_seq_length=128)
    p = models.Pooling(w.get_word_embedding_dimension(), pooling_mode="mean")
    return SentenceTransformer(modules=[w, p])

modelo = cargar_clinico()
print("Modelo clínico cargado.")


/tmp/ipykernel_942/1734970099.py:3: DeprecationWarning: Importing from 'sentence_transformers.models' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.modules' instead.
  from sentence_transformers import SentenceTransformer, models


config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: PlanTL-GOB-ES/bsc-bio-ehr-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

vocab.json:   0%|          | 0.00/1.17M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/521k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

/tmp/ipykernel_942/1734970099.py:8: FutureWarning: The `get_word_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  p = models.Pooling(w.get_word_embedding_dimension(), pooling_mode="mean")


Modelo clínico cargado.


## 2. Anclas semánticas (independientes de las reglas)

Frases que describen el *significado* de cada categoría. NO copian los patrones de las
reglas: son descripciones semánticas, para que el revisor sea un chequeo independiente.

In [2]:
CATEGORIAS = {
    "biopsia_histologia": [
        "se recomienda biopsia", "punción con aguja gruesa para histología",
        "estudio percutáneo con aguja", "toma de muestra tisular",
        "confirmación histológica de la lesión", "punción aspirativa con aguja fina",
        "estudio anatomopatológico de la lesión",
    ],
    "derivacion_oncologica": [
        "derivación a oncología", "referir a mastólogo",
        "valoración por especialista de mama", "interconsulta con cirujano de mama",
    ],
    "estudio_complementario_imagen": [
        "estudio complementario de imagen", "proyecciones adicionales focalizadas",
        "incidencias con magnificación", "completar el estudio mamográfico",
        "ecografía mamaria para recategorización",
    ],
    "correlacion_ecografica": [
        "correlación con ecografía mamaria", "complementar con ultrasonido mamario",
        "ecografía dirigida de la mama", "sonografía mamaria complementaria",
    ],
    "comparacion_estudios_previos": [
        "comparar con estudios previos", "correlacionar con mamografías anteriores",
        "cotejar con exámenes anteriores del archivo",
    ],
    "control_corto_plazo": [
        "control a corto plazo en 6 meses", "repetir mamografía en medio año",
        "seguimiento precoz en tres meses",
    ],
    "control_anual": [
        "control mamográfico anual de rutina", "seguimiento habitual en doce meses",
        "control mamográfico en un año",
    ],
    "criterio_medico": [
        "según criterio del médico tratante", "a consideración del clínico",
    ],
}
nombres_cat = list(CATEGORIAS.keys())
matriz_cat = np.stack([
    modelo.encode(CATEGORIAS[c], normalize_embeddings=True).mean(axis=0)
    for c in nombres_cat
])
print("Anclas embebidas:", len(nombres_cat))

UMBRAL_AMBIGUA = 0.30   # si la mejor similitud < umbral, se marca 'ambigua' (no fuerza)

def clasificar_emb(textos, batch=64):
    emb = modelo.encode(list(textos), normalize_embeddings=True, batch_size=batch)
    sims = emb @ matriz_cat.T
    idx = sims.argmax(axis=1); best = sims.max(axis=1)
    cats = [nombres_cat[i] if best[k] >= UMBRAL_AMBIGUA else "ambigua"
            for k, i in enumerate(idx)]
    return cats, best


Anclas embebidas: 8


## 3. Regla del segundo revisor

Marca discrepancia SOLO si reglas y embedding difieren **y** el desacuerdo no es el ruido
conocido imagen↔imagen.

In [3]:
IMAGEN = {"correlacion_ecografica", "estudio_complementario_imagen"}

def es_discrepancia_util(cat_reglas, cat_emb):
    if cat_reglas == cat_emb:
        return False
    # Excluir el ruido conocido: desacuerdo solo entre las 2 categorías de imagen
    if cat_reglas in IMAGEN and cat_emb in IMAGEN:
        return False
    return True


## 4. Cargar corpus de referencia y medir la tasa de discrepancia

Sube **recomendaciones_clasificadas.csv** (contiene cada recomendación con su
clasificación por reglas).

In [4]:
from google.colab import files
print("Sube recomendaciones_clasificadas.csv ...")
sub = files.upload()
ref = pd.read_csv(io.BytesIO(list(sub.values())[0]))
print(f"{len(ref)} recomendaciones.\n")

cat_emb, sim_emb = clasificar_emb(ref["texto"].astype(str).tolist())
ref["cat_emb"] = cat_emb
ref["sim_emb"] = sim_emb
ref["discrepancia_util"] = [
    es_discrepancia_util(r, e) for r, e in zip(ref["categoria_reglas"], ref["cat_emb"])
]

n_disc = ref["discrepancia_util"].sum()
print(f"Discrepancias TOTALES (reglas != embedding): {(ref['categoria_reglas']!=ref['cat_emb']).sum()}")
print(f"Discrepancias ÚTILES (excluyendo ruido imagen): {n_disc} ({100*n_disc/len(ref):.2f}%)")
print("\n-> Si la tasa útil es baja (1-3%), el revisor es viable (poco ruido).")
print("   Si es alta, generaría demasiadas alarmas para ser práctico.")


Sube recomendaciones_clasificadas.csv ...


Saving recomendaciones_clasificadas.csv to recomendaciones_clasificadas.csv
4349 recomendaciones.

Discrepancias TOTALES (reglas != embedding): 1957
Discrepancias ÚTILES (excluyendo ruido imagen): 1173 (26.97%)

-> Si la tasa útil es baja (1-3%), el revisor es viable (poco ruido).
   Si es alta, generaría demasiadas alarmas para ser práctico.


## 5. ¿Qué son esas discrepancias útiles? (inspección)

In [5]:
disc = ref[ref["discrepancia_util"]]
if len(disc):
    print("Tipos de discrepancia útil (reglas -> embedding):")
    print(disc.groupby(["categoria_reglas","cat_emb"]).size().sort_values(ascending=False).head(12).to_string())
    print("\nEjemplos (¿quién parece tener razón?):")
    for _, r in disc.sample(min(12, len(disc)), random_state=1).iterrows():
        print(f"  '{str(r['texto'])[:68]}'")
        print(f"      reglas={r['categoria_reglas']} | embedding={r['cat_emb']} (sim {r['sim_emb']:.2f}) | BI-RADS={r['birads']}")
else:
    print("No hubo discrepancias útiles.")


Tipos de discrepancia útil (reglas -> embedding):
categoria_reglas               cat_emb                     
correlacion_ecografica         control_anual                   969
control_corto_plazo            control_anual                    47
biopsia_histologia             correlacion_ecografica           43
criterio_medico                control_anual                    35
estudio_complementario_imagen  control_anual                    35
biopsia_histologia             control_anual                    16
correlacion_ecografica         comparacion_estudios_previos     10
estudio_complementario_imagen  comparacion_estudios_previos      9
comparacion_estudios_previos   control_anual                     3
                               correlacion_ecografica            3
criterio_medico                comparacion_estudios_previos      1
                               control_corto_plazo               1

Ejemplos (¿quién parece tener razón?):
  '- se sugiere correlacion con estudios anter

## 6. Cobertura en casos atípicos (el valor real)

Frases externas con sinónimos que las reglas probablemente NO capturan. Aquí el segundo
revisor debería aportar: detectar la categoría correcta donde las reglas fallan.

In [6]:
ATIPICOS = [
    ("se sugiere estudio percutáneo con aguja gruesa", "biopsia_histologia"),
    ("amerita valoración por mastólogo", "derivacion_oncologica"),
    ("procede toma de muestra tisular de la lesión", "biopsia_histologia"),
    ("se aconseja valoración histopatológica", "biopsia_histologia"),
    ("remitir a la unidad de patología mamaria", "derivacion_oncologica"),
    ("repetir el estudio en un semestre", "control_corto_plazo"),
    ("cotejar con las placas antiguas del archivo", "comparacion_estudios_previos"),
]
print("Frase atípica | embedding | (esperada)\n")
ac = 0
textos_at = [t for t,_ in ATIPICOS]
emb_at, sim_at = clasificar_emb(textos_at)
for (frase, esp), ce in zip(ATIPICOS, emb_at):
    ok = ce == esp; ac += ok
    print(f"  {'OK' if ok else 'XX'} '{frase[:52]}' -> {ce}  (esp {esp})")
print(f"\nAciertos del embedding en atípicas: {ac}/{len(ATIPICOS)}")
print("(Estas son las que las reglas basadas en palabras clave fallarían: ahí el revisor aporta.)")


Frase atípica | embedding | (esperada)

  OK 'se sugiere estudio percutáneo con aguja gruesa' -> biopsia_histologia  (esp biopsia_histologia)
  OK 'amerita valoración por mastólogo' -> derivacion_oncologica  (esp derivacion_oncologica)
  OK 'procede toma de muestra tisular de la lesión' -> biopsia_histologia  (esp biopsia_histologia)
  XX 'se aconseja valoración histopatológica' -> derivacion_oncologica  (esp biopsia_histologia)
  OK 'remitir a la unidad de patología mamaria' -> derivacion_oncologica  (esp derivacion_oncologica)
  OK 'repetir el estudio en un semestre' -> control_corto_plazo  (esp control_corto_plazo)
  OK 'cotejar con las placas antiguas del archivo' -> comparacion_estudios_previos  (esp comparacion_estudios_previos)

Aciertos del embedding en atípicas: 6/7
(Estas son las que las reglas basadas en palabras clave fallarían: ahí el revisor aporta.)


## 7. Veredicto

Combina las dos métricas:

- **Tasa de discrepancia útil (sección 4):**
  - Baja (≤3%) → el revisor no satura de alarmas: **viable**.
  - Alta (>10%) → demasiado ruido: no práctico como está.
- **Cobertura en atípicas (sección 6):**
  - Alta (≥6/7) → el revisor detecta redacciones que las reglas fallan: **aporta valor**.
  - Baja → no cubre lo que debía.

**Decisión de integración:**
- Si tasa útil baja **y** cobertura alta → integrar como segundo revisor: cuando discrepa,
  el caso va a revisión humana; robustece informes externos sin tocar la decisión reglada.
- Si tasa útil alta → refinar (más exclusiones) o descartar.
- En todos los casos, el revisor **nunca decide coherencia**: solo señala. La tabla ACR y
  el cotejo siguen siendo la autoridad.

**Nota de honestidad:** sobre el corpus actual el aporte real es limitado (las reglas ya
cubren el 99,8%). El valor se materializa ante informes externos de redacción atípica; la
sección 6 es la evidencia más cercana a ese escenario.
